In [1]:
import pandas as pd
import openai
import os, re
import time, json
 
# Setup
term_input_file = 'Datasets/protein_name_GN.csv'
OPENAI_API_KEY = os.getenv('OPEN_AI_KEY')
openai.api_key = OPENAI_API_KEY
 
# Load data
df = pd.read_csv(term_input_file, header=0)
print(f"Loaded {len(df)} gene symbol records")
 
def get_protein_name_from_gene_symbol(gene_symbol):
    """Get protein name from HUGO gene symbol using GPT-4o."""
    prompt = f"""
    Your task is to identify the correct protein name for a given HUGO gene symbol.
    Respond in the following structured JSON format:
    {{
        "protein_name": "<best matching protein name>"
    }}
    If no relevant protein name is found, return:
    {{
        "protein_name": "None"
    }}
    
    Guidelines for protein names:
    - Provide the most commonly used protein name
    - Use standard nomenclature (e.g., "Tumor protein p53" for TP53)
    - Include descriptive names when available
    - For enzymes, include the enzyme type if it's part of the standard name
    
    Only use well-established protein names. Do not guess or make up names.
    Gene symbol: "{gene_symbol}"
    """
    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names."},
                      {"role": "user", "content": prompt}],
            temperature=0.1
        )
        result = response.choices[0].message.content
        json_content = re.search(r'\{(.*)\}', result, re.DOTALL)
        return json.loads(json_content.group(0))["protein_name"]
    except Exception as e:
        print(f"Error processing '{gene_symbol}': {e}")
        return "Error"
 
# Initialize new columns
df["new_protein_name"] = None
df["match"] = None
 
# Process all gene symbols
total_genes = len(df)
batch_size = 100
 
for i in range(0, total_genes, batch_size):
    print(f"Processing batch {i} to {min(i+batch_size, total_genes)}...")
    batch = df.iloc[i:i+batch_size]
    results = []
    for gene_symbol in batch["GN"]:
        result = get_protein_name_from_gene_symbol(gene_symbol)
        results.append(result)
        time.sleep(0.1)  # Rate limiting
    # Store results
    df.loc[i:i+batch_size-1, "new_protein_name"] = results
    # Create match column (1 if match, 0 if no match)
    for idx in range(i, min(i+batch_size, total_genes)):
        original_protein = df.loc[idx, "protein_name"]
        new_protein = df.loc[idx, "new_protein_name"]
        # Case-insensitive comparison for protein names
        match_result = 1 if (str(original_protein).strip().lower() == str(new_protein).strip().lower()) else 0
        df.loc[idx, "match"] = match_result
 
# Save results to new CSV
output_file = "gene_protein_mapping_results.csv"
df.to_csv(output_file, index=False)
 
# Calculate accuracy
total_processed = len(df)
successful_matches = df['match'].sum()
accuracy = (successful_matches / total_processed) * 100
 
print(f"\n✅ Processing complete!")
print(f"📁 Results saved to: {output_file}")
print(f"📊 Total gene symbols: {total_processed}")
print(f"✅ Successful matches: {successful_matches}")
print(f"📈 Accuracy: {accuracy:.2f}%")

Loaded 3978 gene symbol records
Processing batch 0 to 100...
Processing batch 100 to 200...


KeyboardInterrupt: 

In [8]:
import pandas as pd
import openai
import os, re
import time, json
 
# Setup
term_input_file = './go_terms.csv'
OPENAI_API_KEY = os.getenv('OPEN_AI_KEY')
openai.api_key = OPENAI_API_KEY
 
# Load data
df = pd.read_csv(term_input_file, header=0)
print(f"Loaded {len(df)} GO ID records")
 
def get_go_term_from_id(go_id):
    """Get GO term from GO ID using GPT-4o."""
    prompt = f"""
    Your task is to identify the correct GO (Gene Ontology) term for a given GO ID.
    Respond in the following structured JSON format:
    {{
        "go_term": "<best matching GO term>"
    }}
    If no relevant GO term is found, return:
    {{
        "go_term": "None"
    }}
    
    Guidelines for GO terms:
    - Provide the exact official GO term name
    - Use standard Gene Ontology nomenclature
    - Include the complete term name as it appears in the GO database
    - Do not abbreviate or modify the official term
    
    Only use valid GO terms from the official Gene Ontology database. Do not guess or make up terms.
    GO ID: "{go_id}"
    """
    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "system", "content": "You are a biomedical expert specializing in Gene Ontology (GO) terms and IDs."},
                      {"role": "user", "content": prompt}],
            temperature=0.1
        )
        result = response.choices[0].message.content
        json_content = re.search(r'\{(.*)\}', result, re.DOTALL)
        return json.loads(json_content.group(0))["go_term"]
    except Exception as e:
        print(f"Error processing '{go_id}': {e}")
        return "Error"
 
# Initialize new columns
df["new_go_term"] = None
df["match"] = None
 
# Process all GO IDs
total_ids = len(df)
batch_size = 100
 
for i in range(0, total_ids, batch_size):
    print(f"Processing batch {i} to {min(i+batch_size, total_ids)}...")
    batch = df.iloc[i:i+batch_size]
    results = []
    for go_id in batch["go_id"]:
        result = get_go_term_from_id(go_id)
        results.append(result)
        time.sleep(0.1)  # Rate limiting
    # Store results
    df.loc[i:i+batch_size-1, "new_go_term"] = results
    # Create match column (1 if match, 0 if no match)
    for idx in range(i, min(i+batch_size, total_ids)):
        original_go_term = df.loc[idx, "go_term"]
        new_go_term = df.loc[idx, "new_go_term"]
        # Case-insensitive comparison for GO terms
        match_result = 1 if (str(original_go_term).strip().lower() == str(new_go_term).strip().lower()) else 0
        df.loc[idx, "match"] = match_result
 
# Save results to new CSV
output_file = "go_term_mapping_results.csv"
df.to_csv(output_file, index=False)
 
# Calculate accuracy
total_processed = len(df)
successful_matches = df['match'].sum()
accuracy = (successful_matches / total_processed) * 100
 
print(f"\n✅ Processing complete!")
print(f"📁 Results saved to: {output_file}")
print(f"📊 Total GO IDs: {total_processed}")
print(f"✅ Successful matches: {successful_matches}")
print(f"📈 Accuracy: {accuracy:.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: './go_terms.csv'

In [7]:
pip install openai

  Using cached anyio-4.9.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.10.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.14.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached certifi-2025.4.26-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp313-cp313-win_amd64.whl.metadata (6.9 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/730.3 kB ? eta 